In [1]:
import pandas as pd
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import time
from utils import measure_runtime, extract_columns_from_json

# Load environment variables
load_dotenv()

# Initialize GPT-5 via LangChain
llm = ChatOpenAI(
    model="gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Build messages
messages = [
    SystemMessage(content="You are a helpful AI assistant."),
    HumanMessage(content="Explain RAG in one paragraph.")
]

# Invoke model
start = time.perf_counter()
response = llm.invoke(messages)
end = time.perf_counter()

print(response.content)
print(f"Runtime: {end - start:.4f} seconds")


RAG, which stands for Retrieval-Augmented Generation, is an approach that combines the strengths of retrieval-based and generative models to improve the quality and accuracy of information retrieval in natural language processing tasks. In a RAG system, a retrieval component first searches a large corpus to find relevant documents or passages based on an initial query. Then, a generative model, typically a transformer-based model like GPT, uses the retrieved information to generate more accurate and contextually informed responses. This two-step process allows RAG systems to leverage vast amounts of unstructured data efficiently, enhancing the quality of generated responses by grounding them in specific, retrieved evidence, thus making it particularly useful in applications requiring up-to-date information or domain-specific knowledge.
Runtime: 4.4255 seconds


In [2]:
df = pd.read_csv("data/adserver.csv")

In [3]:
df["status"].unique()

array(['FINISHED', 'PENDING'], dtype=object)

In [4]:
#Preprocess
df = df[df["status"] == "FINISHED"]
df = df[["website", "detect_result", "risk_score", "violation_details"]]

In [5]:
df["detect_result"].value_counts()

detect_result
NON_VIOLATION    4176
VIOLATION        1952
REVIEW            264
Name: count, dtype: int64

In [6]:
df.head()

,website,detect_result,risk_score,violation_details
0,kilo.health,NON_VIOLATION,0.28,"{""violationCategory"":""NON_VIOLATION"",""riskScor..."
1,sugoimart.com,NON_VIOLATION,0.18,"{""violationCategory"":""NON_VIOLATION"",""riskScor..."
2,comfilife.com,NON_VIOLATION,0.22,"{""violationCategory"":""NON_VIOLATION"",""riskScor..."
3,omniluxled.com,NON_VIOLATION,0.38,"{""violationCategory"":""NON_VIOLATION"",""riskScor..."
4,310nutrition.com,NON_VIOLATION,0.22,"{""violationCategory"":""NON_VIOLATION"",""riskScor..."


In [7]:
# df[df["detect_result"] == "VIOLATION"].iloc[0]["violation_details"]
# df[df["detect_result"] == "REVIEW"].iloc[0]["violation_details"]

In [8]:
df[(df["detect_result"] == "VIOLATION") & (df["risk_score"] < 0.5)]


,website,detect_result,risk_score,violation_details
5717,playgames365.net,VIOLATION,0.41,"{""violationCategory"":""NON_VIOLATION"",""riskScor..."


In [9]:
target_cols = ["policyViolations", "summary", "detailedBreakdown"]

df = extract_columns_from_json(
    df,
    source_col="violation_details",
    target_cols=target_cols
)

In [10]:
df["policyViolations"].value_counts()

policyViolations
[]                                        6263
[Weaponry]                                  28
[Sex Toys]                                  23
[Political Content]                         19
[Financial Products]                        17
[Low-Quality Affliate or Review Sites]      17
[Pornography]                                8
[Data Collection]                            7
[Dead Links]                                 3
[Graphic Content]                            2
[Sweepstakes]                                2
[Kratom]                                     1
Name: count, dtype: int64

In [11]:
def first_or_nan(x):
    if isinstance(x, list) and len(x) > 0:
        return x[0]
    return ""

df["policyViolation"] = df["policyViolations"].apply(first_or_nan)


In [12]:
df[df["policyViolation"] != ""]["risk_score"].value_counts()

risk_score
1.00    125
0.76      1
0.82      1
Name: count, dtype: int64

In [13]:
import re
def count_steps(text: str) -> int:
    if not isinstance(text, str):
        return 0

    # 1) Convert escaped newlines/tabs to real whitespace
    text = re.sub(r"\\[nrt]", " ", text)

    # 2) Normalize all whitespace (Unicode-safe)
    text = re.sub(r"\s+", " ", text)

    # 3) Count "Step <number>" patterns
    return len(re.findall(r"\bStep\s+\d+\b", text, flags=re.IGNORECASE))

df["num_steps"] = df["detailedBreakdown"].apply(count_steps)


In [14]:
#Normally, there are 4 steps:

#Step 1 - On-site review
#Step 2 - Web search and external signals
#Step 3 - Risk scoring per required criteria
#Step 4 - Categorization

In [15]:
llm = ChatOpenAI(
    model="gpt-4o",
    model_kwargs={
        "response_format": {"type": "json_object"}
    }
)

In [16]:
df = pd.read_csv("data/final_output.csv")
df = df[["website", "detect_result", "risk_score", "summary", "detailedBreakdown", "risk_features"]]

In [17]:
import ast
def extract_amazon_fields(raw_value):
    """
    Extract amazon_present, amazon_score_impact, amazon_evidence
    from a stringified Python dict.
    """
    if not isinstance(raw_value, str):
        return pd.Series([None, None, None])

    try:
        parsed = ast.literal_eval(raw_value)
        amazon = parsed.get("amazon_presence", {})
        return pd.Series([
            amazon.get("present"),
            amazon.get("score_impact"),
            amazon.get("evidence")
        ])
    except (ValueError, SyntaxError):
        return pd.Series([None, None, None])

In [18]:
df[[
    "amazon_present",
    "amazon_score_impact",
    "amazon_evidence"
]] = df["risk_features"].apply(extract_amazon_fields)

In [19]:
# print(df[df["amazon_score_impact"] == 0.7].iloc[0]["detailedBreakdown"])


In [20]:
#remove outliers
amazon_insight_df = df[df["amazon_score_impact"].abs() < 0.3][["amazon_score_impact", "amazon_evidence"]]

In [21]:
evidences = amazon_insight_df["amazon_evidence"]
scores = amazon_insight_df["amazon_score_impact"]

In [22]:
evidences[0]

'Strong presence via several Kilo brands (ColonBroom, Bioma, Pulsetto) with hundreds to thousands of reviews. Evidence above (Amazon links).'

In [23]:
from src.risk_feature_analyzer.preprocessors.risk_feature_preprocessor import RiskTextPreprocessor
preprocessor = RiskTextPreprocessor()

clean_text = preprocessor.preprocess(
    "Amazon presence with 6,425 reviews and strong Trustpilot rating (4.2/5)."
)

print(clean_text)

[AMAZON] [REVIEWS] amazon presence with reviews_100_plus and strong trustpilot rating trustpilot_positive_rating


In [24]:
processed_evidences = [preprocessor.preprocess(e) for e in evidences]

In [25]:
from typing import List, Dict
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression

class InterpretableRiskModel:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=256,
            stop_words="english",
            min_df=3,
            max_df=0.9
        )
        self.model = LinearRegression()
        self.is_fitted = False

    def fit(self, texts: List[str], scores: List[float]):
        """
        Train model to map text → risk score
        """
        X = self.vectorizer.fit_transform(texts)
        y = np.array(scores)

        self.model.fit(X, y)
        self.is_fitted = True

    def predict_scores(self, texts):
        if not self.is_fitted:
            raise RuntimeError("Model must be fitted first")
    
        X = self.vectorizer.transform(texts)
        preds = self.model.predict(X)
        return preds.tolist()

    def predict_contributions(self, texts: List[str]) -> List[Dict]:
        """
        Returns per-text score contribution
        """
        if not self.is_fitted:
            raise RuntimeError("Model must be fitted first")

        X = self.vectorizer.transform(texts)

        # Linear contribution: X * weights
        contributions = X.multiply(self.model.coef_).sum(axis=1)

        return [
            {
                "text_feature": text,
                "score_contribution": float(contributions[i])
            }
            for i, text in enumerate(texts)
        ]

    def predict_total_risk(self, texts: List[str]) -> Dict:
        """
        Returns interpretable breakdown + final summed risk
        """
        contributions = self.predict_contributions(texts)

        total_risk = sum(item["score_contribution"] for item in contributions)

        return {
            "feature_contributions": contributions,
            "final_risk_score": float(total_risk)
        }


In [26]:
num_train = int(len(evidences) * 0.8)
num_test = len(evidences) - num_train
X_train, y_train, X_test, y_test = processed_evidences[:num_train], scores[:num_train], processed_evidences[num_train:], scores[num_train:]

risk_model = InterpretableRiskModel()
risk_model.fit(X_train, y_train)
y_pred = risk_model.predict_scores(X_test)

mae = np.mean(np.abs(y_test - y_pred))
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.03952164202452867
RMSE: 0.05175094325237302


In [27]:
import numpy as np
import pandas as pd

def extract_top_words(vectorizer, model, top_k=20):
    feature_names = vectorizer.get_feature_names_out()
    coef = model.coef_.ravel()  # shape (n_features,)

    df = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coef
    })

    # Sort
    bad_words = (
        df[df["coefficient"] > 0]
        .sort_values("coefficient", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    good_words = (
        df[df["coefficient"] < 0]
        .sort_values("coefficient")
        .head(top_k)
        .reset_index(drop=True)
    )

    return good_words, bad_words


In [28]:
good_words, bad_words = extract_top_words(
    vectorizer=risk_model.vectorizer,
    model=risk_model.model,  # underlying linear model
    top_k=50
)


In [29]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser
from src.risk_feature_analyzer.extractors.risk_feature_extractor import iterative_rule_discovery
from src.risk_feature_analyzer.prompts.risk_feature_prompt import RULE_MATCHING_PROMPT, RULE_DISCOVERY_PROMPT, amazon_rules

import json

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,   # critical for rule discovery
)


# prompt = ChatPromptTemplate.from_template(RULE_DISCOVERY_PROMPT)

# rule_discovery_chain = prompt | llm | StrOutputParser()

# new_rule_objects, discovered_rules = iterative_rule_discovery(rule_discovery_chain, evidences, [], batch_size = 5)

prompt = ChatPromptTemplate.from_template(RULE_MATCHING_PROMPT)

rule_matching_chain = prompt | llm | StrOutputParser()

response = rule_matching_chain.invoke({
    "rules": amazon_rules,
    "evidence_text": evidences[0],
})



In [76]:
df = pd.read_parquet("checkpoint.parquet")[:2000]

In [84]:
amazon_insight_df = df[["amazon_presence_features", "amazon_score_impact", "amazon_evidence"]].copy()

In [78]:
def parse_rule_names(arr):
    return [item.get("rule_name") for item in arr if isinstance(item, dict)]

In [85]:
amazon_insight_df["amazon_presence_features"] = (
    amazon_insight_df["amazon_presence_features"]
    .apply(parse_rule_names)
)

In [86]:
amazon_insight_df = amazon_insight_df[amazon_insight_df["amazon_presence_features"].str.len() == 0]


In [89]:
amazon_insight_df.iloc[1]["amazon_evidence"]

'Strong presence with many listings/reviews and an official store page.'